In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import torch
import numpy as np
from PIL import Image
import requests

input_text = "<image>What is this an image of?"

In [ ]:
!pip install --upgrade torchao

In [ ]:
from transformers import AutoTokenizer, PaliGemmaForConditionalGeneration, PaliGemmaProcessor
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "TessWOfficial/paligemma-finetuned-rhb"
model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.bfloat16)
processor = PaliGemmaProcessor.from_pretrained(model_id)

In [ ]:
#single image
img_path = "/content/nv.jpg"
input_image = Image.open(img_path)

inputs = processor(text=input_text, images=input_image,
                  padding="longest", do_convert_rgb=True, return_tensors="pt").to("cuda")
model.to(device)
inputs = inputs.to(dtype=model.dtype)

with torch.no_grad():
  output = model.generate(**inputs, max_length=496)

#get predicted classification
print(processor.decode(output[0], skip_special_tokens=True))

In [ ]:
#dir of images
from pathlib import Path

image_dir = Path("/content/test")

model.to(device)

#write results to file
with open("paligemma_results.txt", "a", encoding="utf-8") as f:
    for image_path in image_dir.glob("*.jpg"):
        input_image = Image.open(image_path)

        inputs = processor(
            text=input_text,
            images=input_image,
            padding="longest",
            do_convert_rgb=True,
            return_tensors="pt"
        ).to(device)

        inputs = inputs.to(dtype=model.dtype)

        with torch.no_grad():
            output = model.generate(**inputs, max_length=496)

        decoded = processor.decode(output[0], skip_special_tokens=True)
        f.write(f"{image_path}: {decoded}\n")
